<a href="https://colab.research.google.com/github/DecioVdA/ProveComm/blob/main/TEST_RandomForest_Traffico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [89]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import requests
from io import BytesIO
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_recall_curve
rd_seed = 42  # fissiamo un random seed per riproducibilità e replicabilità
np.random.seed(rd_seed)
random.seed(rd_seed)

rd_seed = 42


In [90]:

import requests
from io import BytesIO

url = "https://github.com/DecioVdA/ProveComm/raw/refs/heads/main/DataSetAddestramento.xlsx"
response = requests.get(url)
df = pd.read_excel(BytesIO(response.content))


In [91]:
df.head()

,Data,GiornoSettimana,Scolastiche_VDA,Scolastiche_FRA,Scolastiche_CH,Festivo_ITA,Festivo_FRA,Festivo_CH,Dispo_TU,Dispo_Fre,TransitiFRA
0,2024-01-01,Lunedi,1,ABC,nullo,FE,FE,FE,1.0,1.0,2274
1,2024-01-02,Martedi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,3160
2,2024-01-03,Mercoledi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2667
3,2024-01-04,Giovedi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2633
4,2024-01-05,Venerdi,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2608


In [92]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Data             366 non-null    datetime64[ns]
 1   GiornoSettimana  366 non-null    object        
 2   Scolastiche_VDA  366 non-null    int64         
 3   Scolastiche_FRA  366 non-null    object        
 4   Scolastiche_CH   366 non-null    object        
 5   Festivo_ITA      366 non-null    object        
 6   Festivo_FRA      366 non-null    object        
 7   Festivo_CH       366 non-null    object        
 8   Dispo_TU         366 non-null    float64       
 9   Dispo_Fre        366 non-null    float64       
 10  TransitiFRA      366 non-null    int64         
dtypes: datetime64[ns](1), float64(2), int64(2), object(6)
memory usage: 31.6+ KB


In [93]:
df['Data'] = pd.to_datetime(df['Data'])
df['Day'] = df['Data'].dt.day # giorno del mese
df['Day_of_week'] = df['Data'].dt.day_of_week
df['Day_of_year'] = df['Data'].dt.day_of_year
df['Month'] = df['Data'].dt.month


In [94]:
df = df.drop(['Data'], axis = 1)
df = df.drop(['GiornoSettimana'], axis = 1)
df


,Scolastiche_VDA,Scolastiche_FRA,Scolastiche_CH,Festivo_ITA,Festivo_FRA,Festivo_CH,Dispo_TU,Dispo_Fre,TransitiFRA,Day,Day_of_week,Day_of_year,Month
0,1,ABC,nullo,FE,FE,FE,1.0,1.0,2274,1,0,1,1
1,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,3160,2,1,2,1
2,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2667,3,2,3,1
3,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2633,4,3,4,1
4,1,ABC,nullo,nullo,nullo,nullo,1.0,1.0,2608,5,4,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,1,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2789,27,4,362,12
362,1,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,3095,28,5,363,12
363,1,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2673,29,6,364,12
364,1,ABC,GVZ,nullo,nullo,nullo,1.0,1.0,2663,30,0,365,12


In [95]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Scolastiche_VDA  366 non-null    int64  
 1   Scolastiche_FRA  366 non-null    object 
 2   Scolastiche_CH   366 non-null    object 
 3   Festivo_ITA      366 non-null    object 
 4   Festivo_FRA      366 non-null    object 
 5   Festivo_CH       366 non-null    object 
 6   Dispo_TU         366 non-null    float64
 7   Dispo_Fre        366 non-null    float64
 8   TransitiFRA      366 non-null    int64  
 9   Day              366 non-null    int32  
 10  Day_of_week      366 non-null    int32  
 11  Day_of_year      366 non-null    int32  
 12  Month            366 non-null    int32  
dtypes: float64(2), int32(4), int64(2), object(5)
memory usage: 31.6+ KB


In [96]:
df['Scolastiche_VDA'] = df['Scolastiche_VDA'].replace({1: 'Si', 0:'No'})

In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Scolastiche_VDA  366 non-null    object 
 1   Scolastiche_FRA  366 non-null    object 
 2   Scolastiche_CH   366 non-null    object 
 3   Festivo_ITA      366 non-null    object 
 4   Festivo_FRA      366 non-null    object 
 5   Festivo_CH       366 non-null    object 
 6   Dispo_TU         366 non-null    float64
 7   Dispo_Fre        366 non-null    float64
 8   TransitiFRA      366 non-null    int64  
 9   Day              366 non-null    int32  
 10  Day_of_week      366 non-null    int32  
 11  Day_of_year      366 non-null    int32  
 12  Month            366 non-null    int32  
dtypes: float64(2), int32(4), int64(1), object(6)
memory usage: 31.6+ KB


In [98]:
df['Day_of_week'] = df['Day_of_week'].astype(str)

In [99]:
categoriche = ['Scolastiche_VDA', 'Scolastiche_FRA', 'Scolastiche_CH', 'Festivo_ITA', 'Festivo_FRA', 'Festivo_CH', 'Day_of_week']
numeriche = list(set(df.columns)- set(df[categoriche]) - {'TransitiFRA'})

In [100]:
list(categoriche)
list(numeriche)

['Month', 'Day_of_year', 'Dispo_TU', 'Day', 'Dispo_Fre']

In [101]:
fig = px.line(df, x = 'Day_of_year', y = 'TransitiFRA', title = 'Andamento del Traffico')
fig.show()

#### Separazione del dataset

In [111]:
X = df.drop(['TransitiFRA'], axis =1)
y = df['TransitiFRA']

In [112]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=rd_seed)
X_train.shape[0] + X_test.shape[0] == df.shape[0]

True

In [113]:
encoder = OneHotEncoder().fit(X_train[categoriche])
X_train_cat_enc = encoder.transform(X_train[categoriche]).todense().astype(int)
X_train_cat_enc = pd.DataFrame(X_train_cat_enc, columns = encoder.get_feature_names_out(categoriche), index = X_train.index)
X_test_cat_enc = encoder.transform(X_test[categoriche]).todense().astype(int)
X_test_cat_enc = pd.DataFrame(X_test_cat_enc, columns = encoder.get_feature_names_out(categoriche), index = X_test.index)
X_train_cat_enc

,Scolastiche_VDA_No,Scolastiche_VDA_Si,Scolastiche_FRA_AB,Scolastiche_FRA_ABC,Scolastiche_FRA_AC,Scolastiche_FRA_B,Scolastiche_FRA_C,Scolastiche_FRA_nullo,Scolastiche_CH_G,Scolastiche_CH_GV,...,Festivo_FRA_nullo,Festivo_CH_FE,Festivo_CH_nullo,Day_of_week_0,Day_of_week_1,Day_of_week_2,Day_of_week_3,Day_of_week_4,Day_of_week_5,Day_of_week_6
268,1,0,0,0,0,0,0,1,0,0,...,1,0,1,0,0,1,0,0,0,0
231,0,1,0,1,0,0,0,0,0,0,...,1,0,1,1,0,0,0,0,0,0
157,0,1,0,0,0,0,0,1,0,0,...,1,0,1,0,0,0,1,0,0,0
19,1,0,0,0,0,0,0,1,0,0,...,1,0,1,0,0,0,0,0,1,0
147,1,0,0,0,0,0,0,1,0,0,...,1,0,1,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,1,0,0,0,0,0,0,1,0,0,...,1,0,1,0,1,0,0,0,0,0
106,1,0,0,0,1,0,0,0,0,0,...,1,0,1,0,1,0,0,0,0,0
270,1,0,0,0,0,0,0,1,0,0,...,1,0,1,0,0,0,0,1,0,0
348,1,0,0,0,0,0,0,1,0,0,...,1,0,1,0,0,0,0,0,1,0


In [105]:
list(X_train_cat_enc.columns)

['Scolastiche_VDA_No',
 'Scolastiche_VDA_Si',
 'Scolastiche_FRA_AB',
 'Scolastiche_FRA_ABC',
 'Scolastiche_FRA_AC',
 'Scolastiche_FRA_B',
 'Scolastiche_FRA_C',
 'Scolastiche_FRA_nullo',
 'Scolastiche_CH_G',
 'Scolastiche_CH_GV',
 'Scolastiche_CH_GVZ',
 'Scolastiche_CH_VZ',
 'Scolastiche_CH_Z',
 'Scolastiche_CH_nullo',
 'Festivo_ITA_FE',
 'Festivo_ITA_RI',
 'Festivo_ITA_nullo',
 'Festivo_FRA_FE',
 'Festivo_FRA_nullo',
 'Festivo_CH_FE',
 'Festivo_CH_nullo',
 'Day_of_week_0',
 'Day_of_week_1',
 'Day_of_week_2',
 'Day_of_week_3',
 'Day_of_week_4',
 'Day_of_week_5',
 'Day_of_week_6']

In [106]:
X_train = pd.concat([X_train[numeriche], X_train_cat_enc], axis=1)
X_test = pd.concat([X_test[numeriche], X_test_cat_enc], axis=1)
X_train

,Month,Day_of_year,Dispo_TU,Day,Dispo_Fre,Scolastiche_VDA_No,Scolastiche_VDA_Si,Scolastiche_FRA_AB,Scolastiche_FRA_ABC,Scolastiche_FRA_AC,...,Festivo_FRA_nullo,Festivo_CH_FE,Festivo_CH_nullo,Day_of_week_0,Day_of_week_1,Day_of_week_2,Day_of_week_3,Day_of_week_4,Day_of_week_5,Day_of_week_6
268,9,269,0.0,25,1.000000,1,0,0,0,0,...,1,0,1,0,0,1,0,0,0,0
231,8,232,1.0,19,1.000000,0,1,0,1,0,...,1,0,1,1,0,0,0,0,0,0
157,6,158,1.0,6,1.000000,0,1,0,0,0,...,1,0,1,0,0,0,1,0,0,0
19,1,20,1.0,20,1.000000,1,0,0,0,0,...,1,0,1,0,0,0,0,0,1,0
147,5,148,1.0,27,1.000000,1,0,0,0,0,...,1,0,1,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,3,72,1.0,12,0.958333,1,0,0,0,0,...,1,0,1,0,1,0,0,0,0,0
106,4,107,1.0,16,1.000000,1,0,0,0,1,...,1,0,1,0,1,0,0,0,0,0
270,9,271,0.0,27,1.000000,1,0,0,0,0,...,1,0,1,0,0,0,0,1,0,0
348,12,349,0.0,14,1.000000,1,0,0,0,0,...,1,0,1,0,0,0,0,0,1,0


In [107]:
model = RandomForestClassifier(random_state = rd_seed)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [108]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [109]:
print('Prestazione sul training set: {0:0.2%}'. format(accuracy_score(y_train, y_pred_train)))
print('Prestazione sul test set: {0:0.2%}'. format(accuracy_score(y_test, y_pred_test)))

Prestazione sul training set: 100.00%
Prestazione sul test set: 27.27%


In [110]:
y_pred_train

array([   0, 3226, 2175, 2166, 1722, 2359, 3087, 2817,    0, 2447,    0,
       3191, 2696, 2137,    0, 3357,    0,    0, 1803, 3015, 2884, 2728,
       1928, 2329, 2023,    0,    0, 2117, 2186, 3077, 3025, 6709, 5330,
       2844, 3094,    0, 2257, 3687, 2746, 1911, 2045, 3063, 2016,    0,
       2914,    0, 2384, 2166, 2228, 2673, 2112, 2998, 2196,    0,    0,
          0, 2244, 1747,    0, 2739,    0, 3337,    0, 2652,    0,  139,
          0, 3649,    0, 3689, 2299, 2521, 3321, 4374, 1853, 3532, 3692,
       3030, 2899,    0, 2547,    0,    0, 2418, 3341,    0, 1667, 2340,
          0, 2667, 1769, 3193,    0,    0, 2323,    0, 3945,    0, 3109,
       2855,    0, 2188, 1879,    0, 2157,    0, 3595, 1803, 3260,    0,
       2477,    0, 2506,    0, 2234, 4120, 1945, 2506, 2145, 1715, 2073,
       2562, 1911, 3122, 2161, 3594, 3988, 2055,    0, 2582,    0, 3090,
          0, 3120,  273, 2940, 2147, 3373,    0, 4171, 2526, 3562,    0,
       1613,    0,    0, 2608,    0,    0, 2141, 31

plt.plot(y_pred_train)